
# RetailIQ — Customer Segmentation, Cohort Retention & Predictive Customer Value

## Objective

This project turns raw e-commerce transactions into actionable customer analytics through three connected analyses:

1. **Customer segmentation** using RFM features and K-Means clustering.
2. **Cohort retention analysis** to understand how customer activity changes after acquisition.
3. **90-day future spend prediction** using supervised machine learning.

The goal is to demonstrate a complete data-science workflow:

**Data cleaning → EDA → feature engineering → unsupervised ML → retention analysis → supervised ML → model evaluation → business interpretation**

### Dataset
UCI Online Retail II, containing two years of transaction-level data.

### Main techniques
- Pandas / NumPy
- RFM analysis
- StandardScaler
- K-Means
- Elbow Method and Silhouette Score
- PCA
- Cohort retention
- Linear Regression
- Random Forest
- XGBoost
- MAE / RMSE / R²
- Feature importance

> This notebook is designed for Google Colab and uses CPU only.


## 1. Setup

In [ ]:

!pip -q install openpyxl xgboost


In [ ]:

import os
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RANDOM_STATE = 42



## 2. Upload the dataset

Upload **`online_retail_II.xlsx`** when prompted.

If the workbook is already present in the Colab working directory, the upload step is skipped.


In [ ]:

DATA_FILE = "online_retail_II.xlsx"

try:
    from google.colab import files

    if not os.path.exists(DATA_FILE):
        print(f"Please upload {DATA_FILE}")
        uploaded = files.upload()

        if DATA_FILE not in uploaded:
            xlsx_files = [name for name in uploaded if name.lower().endswith(".xlsx")]
            if not xlsx_files:
                raise FileNotFoundError("No .xlsx workbook was uploaded.")
            DATA_FILE = xlsx_files[0]

except ImportError:
    if not os.path.exists(DATA_FILE):
        raise FileNotFoundError(
            f"Place '{DATA_FILE}' in the current working directory before running the notebook."
        )

print("Using dataset:", DATA_FILE)


## 3. Load both years of transaction data

In [ ]:

excel_file = pd.ExcelFile(DATA_FILE)
print("Sheets found:", excel_file.sheet_names)

required_sheets = ["Year 2009-2010", "Year 2010-2011"]
missing_sheets = [sheet for sheet in required_sheets if sheet not in excel_file.sheet_names]

if missing_sheets:
    raise ValueError(f"Expected sheet(s) not found: {missing_sheets}")

frames = []

for sheet in required_sheets:
    temp = pd.read_excel(DATA_FILE, sheet_name=sheet)
    temp["SourceYear"] = sheet
    frames.append(temp)

df_raw = pd.concat(frames, ignore_index=True)

print("Combined shape:", df_raw.shape)
display(df_raw.head())



## 4. Initial data audit

We inspect the raw data before cleaning because:
- missing customer IDs prevent customer-level aggregation;
- cancelled invoices and negative quantities often represent returns/cancellations;
- zero or negative prices should not be treated as normal purchases.


In [ ]:

print("Rows:", f"{len(df_raw):,}")
print("Columns:", df_raw.shape[1])

print("\nColumn names:")
print(df_raw.columns.tolist())

print("\nData types:")
display(df_raw.dtypes.to_frame("dtype"))

missing = pd.DataFrame({
    "Missing Count": df_raw.isna().sum(),
    "Missing %": df_raw.isna().mean() * 100
}).sort_values("Missing Count", ascending=False)

print("\nMissing values:")
display(missing)

print("Duplicate rows:", f"{df_raw.duplicated().sum():,}")


In [ ]:

print("Date range:", df_raw["InvoiceDate"].min(), "to", df_raw["InvoiceDate"].max())
print("Unique invoices:", f"{df_raw['Invoice'].nunique():,}")
print("Unique customers:", f"{df_raw['Customer ID'].nunique(dropna=True):,}")
print("Unique stock codes:", f"{df_raw['StockCode'].nunique():,}")
print("Countries:", f"{df_raw['Country'].nunique():,}")

audit = df_raw.copy()
audit["Invoice"] = audit["Invoice"].astype(str)

quality_checks = pd.Series({
    "Cancelled invoice lines": audit["Invoice"].str.startswith("C", na=False).sum(),
    "Quantity <= 0": (audit["Quantity"] <= 0).sum(),
    "Price <= 0": (audit["Price"] <= 0).sum(),
    "Missing Customer ID": audit["Customer ID"].isna().sum(),
    "Missing Description": audit["Description"].isna().sum(),
    "Duplicate rows": audit.duplicated().sum(),
})

display(quality_checks.to_frame("Rows"))



## 5. Data cleaning

For customer analytics we retain successful, positive-value purchases linked to a known customer.

Cleaning rules:
1. Remove rows with missing `Customer ID`.
2. Remove cancelled invoices.
3. Keep `Quantity > 0`.
4. Keep `Price > 0`.
5. Remove exact duplicates.
6. Create transaction-level `Revenue = Quantity × Price`.

The untouched raw table remains available as `df_raw`.


In [ ]:

df = df_raw.copy()

df["Invoice"] = df["Invoice"].astype(str)
df = df.dropna(subset=["Customer ID"]).copy()
df = df[~df["Invoice"].str.startswith("C", na=False)].copy()
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)].copy()
df = df.drop_duplicates().copy()

df["Customer ID"] = df["Customer ID"].astype(int).astype(str)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Revenue"] = df["Quantity"] * df["Price"]

print("Raw rows:", f"{len(df_raw):,}")
print("Clean purchase rows:", f"{len(df):,}")
print("Rows retained:", f"{len(df) / len(df_raw) * 100:.1f}%")
print("Customers retained:", f"{df['Customer ID'].nunique():,}")
print("Invoices retained:", f"{df['Invoice'].nunique():,}")


## 6. Exploratory Data Analysis

In [ ]:

summary = pd.Series({
    "Transactions": len(df),
    "Customers": df["Customer ID"].nunique(),
    "Invoices": df["Invoice"].nunique(),
    "Products": df["StockCode"].nunique(),
    "Countries": df["Country"].nunique(),
    "Total Revenue": df["Revenue"].sum(),
    "Average Order Value": df.groupby("Invoice")["Revenue"].sum().mean(),
})

display(summary.to_frame("Value"))


### 6.1 Revenue by country

In [ ]:

country_summary = (
    df.groupby("Country")
      .agg(
          Revenue=("Revenue", "sum"),
          Customers=("Customer ID", "nunique"),
          Invoices=("Invoice", "nunique"),
      )
      .sort_values("Revenue", ascending=False)
)

display(country_summary.head(10))

top_countries = country_summary.head(10).sort_values("Revenue")

plt.figure(figsize=(10, 6))
plt.barh(top_countries.index, top_countries["Revenue"])
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


### 6.2 Monthly revenue trend

In [ ]:

monthly_revenue = (
    df.set_index("InvoiceDate")
      .resample("MS")["Revenue"]
      .sum()
)

plt.figure(figsize=(12, 5))
plt.plot(monthly_revenue.index, monthly_revenue.values, marker="o", markersize=3)
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


### 6.3 Top products by revenue

In [ ]:

product_summary = (
    df.groupby(["StockCode", "Description"], dropna=False)
      .agg(
          Revenue=("Revenue", "sum"),
          Quantity=("Quantity", "sum"),
      )
      .sort_values("Revenue", ascending=False)
      .head(10)
      .reset_index()
)

display(product_summary)



# Part A — Customer Segmentation

## 7. RFM feature engineering

We aggregate the transaction table into one row per customer.

- **Recency** = days since most recent purchase.
- **Frequency** = number of distinct invoices.
- **Monetary** = total historical spend.


In [ ]:

analysis_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print("Analysis date:", analysis_date.date())

rfm = (
    df.groupby("Customer ID")
      .agg(
          Recency=("InvoiceDate", lambda x: (analysis_date - x.max()).days),
          Frequency=("Invoice", "nunique"),
          Monetary=("Revenue", "sum"),
          FirstPurchase=("InvoiceDate", "min"),
          LastPurchase=("InvoiceDate", "max"),
      )
      .reset_index()
)

rfm["CustomerTenureDays"] = (analysis_date - rfm["FirstPurchase"]).dt.days

print("Customer-level RFM shape:", rfm.shape)
display(rfm.head())


In [ ]:

display(rfm[["Recency", "Frequency", "Monetary"]].describe().T)

for column in ["Recency", "Frequency", "Monetary"]:
    plt.figure(figsize=(8, 4))
    plt.hist(rfm[column], bins=50)
    plt.title(f"Distribution of {column}")
    plt.xlabel(column)
    plt.ylabel("Customers")
    plt.tight_layout()
    plt.show()



## 8. Prepare RFM features for K-Means

K-Means is distance-based, so we:
1. clip each variable at its 1st and 99th percentiles to reduce extreme outlier influence;
2. use `log1p` to reduce strong right-skew;
3. standardise the features so one scale does not dominate Euclidean distance.


In [ ]:

rfm_model = rfm[["Recency", "Frequency", "Monetary"]].copy()

for col in rfm_model.columns:
    lower = rfm_model[col].quantile(0.01)
    upper = rfm_model[col].quantile(0.99)
    rfm_model[col] = rfm_model[col].clip(lower=lower, upper=upper)

rfm_log = np.log1p(rfm_model)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

rfm_scaled_df = pd.DataFrame(
    rfm_scaled,
    columns=["Recency_scaled", "Frequency_scaled", "Monetary_scaled"],
    index=rfm.index,
)

display(rfm_scaled_df.head())



## 9. Evaluate the number of clusters

We inspect K from 2 to 8 using:
- **Inertia / Elbow Method**
- **Silhouette Score**

For this project we use **K=4** because it produces richer and more actionable customer groups.  
If another K achieves a higher silhouette score, we report that honestly rather than claiming K=4 is mathematically optimal.


In [ ]:

k_values = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(rfm_scaled)

    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, labels))

k_evaluation = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette Score": silhouette_scores,
})

display(k_evaluation)

best_k_by_silhouette = int(
    k_evaluation.loc[k_evaluation["Silhouette Score"].idxmax(), "K"]
)

print("Best K by silhouette score:", best_k_by_silhouette)


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(k_evaluation["K"], k_evaluation["Inertia"], marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(k_evaluation["K"])
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(k_evaluation["K"], k_evaluation["Silhouette Score"], marker="o")
plt.title("Silhouette Score by K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.xticks(k_evaluation["K"])
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 10. Train the final K-Means model

In [ ]:

SELECTED_K = 4

kmeans = KMeans(
    n_clusters=SELECTED_K,
    random_state=RANDOM_STATE,
    n_init=20,
)

rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

final_silhouette = silhouette_score(rfm_scaled, rfm["Cluster"])

print("Selected K:", SELECTED_K)
print("Final silhouette score:", round(final_silhouette, 4))
print("\nCustomers per cluster:")
display(rfm["Cluster"].value_counts().sort_index().to_frame("Customers"))


## 11. Cluster profiling and business labels

In [ ]:

cluster_summary = (
    rfm.groupby("Cluster")
       .agg(
           Customers=("Customer ID", "count"),
           AvgRecency=("Recency", "mean"),
           MedianRecency=("Recency", "median"),
           AvgFrequency=("Frequency", "mean"),
           MedianFrequency=("Frequency", "median"),
           AvgMonetary=("Monetary", "mean"),
           MedianMonetary=("Monetary", "median"),
           TotalRevenue=("Monetary", "sum"),
       )
       .reset_index()
)

cluster_summary["CustomerShare_%"] = (
    cluster_summary["Customers"] / cluster_summary["Customers"].sum() * 100
)

cluster_summary["RevenueShare_%"] = (
    cluster_summary["TotalRevenue"] / cluster_summary["TotalRevenue"].sum() * 100
)

display(cluster_summary.sort_values("AvgMonetary", ascending=False))


In [ ]:

# Create reproducible labels from the observed profiles rather than cluster IDs.

profile = cluster_summary[
    ["Cluster", "AvgRecency", "AvgFrequency", "AvgMonetary"]
].copy()

profile["RecencyScore"] = profile["AvgRecency"].rank(
    pct=True, ascending=False
)
profile["FrequencyScore"] = profile["AvgFrequency"].rank(
    pct=True, ascending=True
)
profile["MonetaryScore"] = profile["AvgMonetary"].rank(
    pct=True, ascending=True
)

profile["OverallScore"] = profile[
    ["RecencyScore", "FrequencyScore", "MonetaryScore"]
].mean(axis=1)

high_cluster = int(
    profile.loc[profile["OverallScore"].idxmax(), "Cluster"]
)
low_cluster = int(
    profile.loc[profile["OverallScore"].idxmin(), "Cluster"]
)

remaining = profile[
    ~profile["Cluster"].isin([high_cluster, low_cluster])
].copy()

recent_cluster = int(
    remaining.loc[remaining["AvgRecency"].idxmin(), "Cluster"]
)

lapsed_cluster = int(
    remaining.loc[remaining["Cluster"] != recent_cluster, "Cluster"].iloc[0]
)

cluster_to_segment = {
    high_cluster: "High-Value Loyal",
    recent_cluster: "Recent / Developing",
    lapsed_cluster: "Lapsed Valuable Customers",
    low_cluster: "Low-Engagement / Dormant",
}

rfm["Segment"] = rfm["Cluster"].map(cluster_to_segment)
cluster_summary["Segment"] = cluster_summary["Cluster"].map(cluster_to_segment)

cluster_summary = cluster_summary[
    [
        "Cluster", "Segment", "Customers", "CustomerShare_%",
        "AvgRecency", "AvgFrequency", "AvgMonetary",
        "TotalRevenue", "RevenueShare_%"
    ]
].sort_values("AvgMonetary", ascending=False)

display(cluster_summary)


## 12. Segment profile visualisation

In [ ]:

plot_summary = (
    rfm.groupby("Segment")[["Recency", "Frequency", "Monetary"]]
       .mean()
)

for metric in ["Recency", "Frequency", "Monetary"]:
    values = plot_summary[metric].sort_values()

    plt.figure(figsize=(9, 4))
    plt.barh(values.index, values.values)
    plt.title(f"Average {metric} by Customer Segment")
    plt.xlabel(metric)
    plt.ylabel("Segment")
    plt.tight_layout()
    plt.show()



## 13. PCA visualisation

PCA is used only to project the three standardised RFM dimensions into two dimensions for visualisation.  
It does **not** create the clusters.


In [ ]:

pca = PCA(n_components=2)
pca_components = pca.fit_transform(rfm_scaled)

rfm["PCA1"] = pca_components[:, 0]
rfm["PCA2"] = pca_components[:, 1]

print(
    "Variance explained by PCA1 + PCA2:",
    f"{pca.explained_variance_ratio_.sum() * 100:.1f}%"
)

plt.figure(figsize=(10, 6))

for cluster in sorted(rfm["Cluster"].unique()):
    subset = rfm[rfm["Cluster"] == cluster]

    plt.scatter(
        subset["PCA1"],
        subset["PCA2"],
        s=18,
        alpha=0.55,
        label=f"Cluster {cluster}: {cluster_to_segment[cluster]}",
    )

plt.title("Customer Segments — PCA Projection")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(fontsize=8)
plt.grid(alpha=0.15)
plt.tight_layout()
plt.show()



# Part B — Cohort Retention Analysis

## 14. Build monthly acquisition cohorts

A customer's **cohort month** is the month of their first purchase.

We then calculate the percentage of the original cohort that remains active in each subsequent month.


In [ ]:

cohort_df = df[
    ["Customer ID", "Invoice", "InvoiceDate"]
].copy()

cohort_df["InvoiceMonth"] = (
    cohort_df["InvoiceDate"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

first_purchase_month = (
    cohort_df.groupby("Customer ID")["InvoiceMonth"]
    .min()
    .rename("CohortMonth")
)

cohort_df = cohort_df.merge(
    first_purchase_month,
    on="Customer ID",
    how="left"
)

cohort_df["CohortIndex"] = (
    (cohort_df["InvoiceMonth"].dt.year - cohort_df["CohortMonth"].dt.year) * 12
    + (cohort_df["InvoiceMonth"].dt.month - cohort_df["CohortMonth"].dt.month)
)

cohort_df.head()


In [ ]:

cohort_counts = (
    cohort_df.groupby(["CohortMonth", "CohortIndex"])["Customer ID"]
    .nunique()
    .reset_index(name="ActiveCustomers")
)

cohort_matrix = cohort_counts.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="ActiveCustomers"
)

cohort_sizes = cohort_matrix[0]

retention = cohort_matrix.divide(cohort_sizes, axis=0)

print("Customer retention matrix:")
display(retention.round(3))


### 14.1 Retention heatmap — first 12 months

In [ ]:

retention_12 = retention.iloc[:, :12]

fig, ax = plt.subplots(figsize=(13, 8))
image = ax.imshow(retention_12.values, aspect="auto", vmin=0, vmax=1)

ax.set_title("Monthly Customer Retention by Acquisition Cohort")
ax.set_xlabel("Months Since First Purchase")
ax.set_ylabel("Acquisition Cohort")

ax.set_xticks(range(retention_12.shape[1]))
ax.set_xticklabels(retention_12.columns)

ax.set_yticks(range(retention_12.shape[0]))
ax.set_yticklabels(
    [d.strftime("%Y-%m") for d in retention_12.index]
)

for i in range(retention_12.shape[0]):
    for j in range(retention_12.shape[1]):
        value = retention_12.iloc[i, j]

        if pd.notna(value):
            ax.text(
                j, i,
                f"{value:.0%}",
                ha="center",
                va="center",
                fontsize=7,
            )

fig.colorbar(image, ax=ax, label="Retention Rate")
plt.tight_layout()
plt.show()



# Part C — Predict Future 90-Day Customer Spend

## 15. Define a leakage-safe prediction problem

We use the final 90 days of the dataset as the **target window**.

For each customer:
- features are calculated only from transactions **before or on the cut-off date**;
- target = revenue generated during the **following 90 days**.

This prevents future information from leaking into the model features.


In [ ]:

prediction_end = df["InvoiceDate"].max()
cutoff_date = prediction_end - pd.Timedelta(days=90)

historical = df[df["InvoiceDate"] <= cutoff_date].copy()
future = df[df["InvoiceDate"] > cutoff_date].copy()

print("Historical feature window:")
print(historical["InvoiceDate"].min(), "to", historical["InvoiceDate"].max())

print("\nFuture target window:")
print(future["InvoiceDate"].min(), "to", future["InvoiceDate"].max())

print("\nHistorical customers:", f"{historical['Customer ID'].nunique():,}")
print("Future purchasing customers:", f"{future['Customer ID'].nunique():,}")



## 16. Predictive feature engineering

We create customer features from the historical window only:

- Recency
- Frequency
- Monetary
- Average order value
- Total quantity purchased
- Number of unique products
- Customer tenure
- Spend and order activity in the previous 30 and 90 days
- Recent spend/order ratios


In [ ]:

cutoff_analysis_date = cutoff_date + pd.Timedelta(days=1)

customer_features = (
    historical.groupby("Customer ID")
    .agg(
        Recency=("InvoiceDate", lambda x: (cutoff_analysis_date - x.max()).days),
        Frequency=("Invoice", "nunique"),
        Monetary=("Revenue", "sum"),
        TotalQuantity=("Quantity", "sum"),
        UniqueProducts=("StockCode", "nunique"),
        FirstPurchase=("InvoiceDate", "min"),
    )
    .reset_index()
)

customer_features["CustomerTenureDays"] = (
    cutoff_analysis_date - customer_features["FirstPurchase"]
).dt.days.clip(lower=1)

customer_features["AverageOrderValue"] = (
    customer_features["Monetary"] / customer_features["Frequency"]
)

last_30_start = cutoff_date - pd.Timedelta(days=30)
last_90_start = cutoff_date - pd.Timedelta(days=90)

hist_30 = historical[
    historical["InvoiceDate"] > last_30_start
]

hist_90 = historical[
    historical["InvoiceDate"] > last_90_start
]

features_30 = (
    hist_30.groupby("Customer ID")
    .agg(
        SpendLast30=("Revenue", "sum"),
        OrdersLast30=("Invoice", "nunique"),
    )
    .reset_index()
)

features_90 = (
    hist_90.groupby("Customer ID")
    .agg(
        SpendLast90=("Revenue", "sum"),
        OrdersLast90=("Invoice", "nunique"),
    )
    .reset_index()
)

customer_features = customer_features.merge(
    features_30, on="Customer ID", how="left"
)

customer_features = customer_features.merge(
    features_90, on="Customer ID", how="left"
)

for col in [
    "SpendLast30", "OrdersLast30",
    "SpendLast90", "OrdersLast90"
]:
    customer_features[col] = customer_features[col].fillna(0)

customer_features["RecentSpendRatio"] = (
    customer_features["SpendLast30"]
    / (customer_features["SpendLast90"] + 1.0)
)

customer_features["RecentOrderRatio"] = (
    customer_features["OrdersLast30"]
    / (customer_features["OrdersLast90"] + 1.0)
)

customer_features.drop(
    columns=["FirstPurchase"],
    inplace=True
)

display(customer_features.head())


## 17. Create the future 90-day target

In [ ]:

future_spend = (
    future.groupby("Customer ID")["Revenue"]
    .sum()
    .rename("Future90DaySpend")
    .reset_index()
)

model_data = customer_features.merge(
    future_spend,
    on="Customer ID",
    how="left"
)

model_data["Future90DaySpend"] = (
    model_data["Future90DaySpend"].fillna(0)
)

print("Modelling customers:", f"{len(model_data):,}")
print(
    "Customers with > 0 future spend:",
    f"{(model_data['Future90DaySpend'] > 0).sum():,}"
)

display(model_data.head())


In [ ]:

target_summary = model_data["Future90DaySpend"].describe()

print("Future 90-day spend distribution:")
display(target_summary.to_frame("Future90DaySpend"))

plt.figure(figsize=(9, 4))
plt.hist(
    np.log1p(model_data["Future90DaySpend"]),
    bins=50,
)
plt.title("Distribution of log(1 + Future 90-Day Spend)")
plt.xlabel("log(1 + Spend)")
plt.ylabel("Customers")
plt.tight_layout()
plt.show()



## 18. Train / test split

All customers share the same historical cut-off date, so we split customers into training and test sets.

The target is strongly right-skewed, so the ML models are trained on:

\[
\log(1 + 	ext{FutureSpend})
\]

Predictions are converted back to the original revenue scale before evaluation.


In [ ]:

feature_columns = [
    "Recency",
    "Frequency",
    "Monetary",
    "TotalQuantity",
    "UniqueProducts",
    "CustomerTenureDays",
    "AverageOrderValue",
    "SpendLast30",
    "OrdersLast30",
    "SpendLast90",
    "OrdersLast90",
    "RecentSpendRatio",
    "RecentOrderRatio",
]

X = model_data[feature_columns].copy()
y = model_data["Future90DaySpend"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

y_train_log = np.log1p(y_train)

print("Training customers:", len(X_train))
print("Test customers:", len(X_test))



## 19. Train and compare regression models

Models:
1. Mean baseline
2. Linear Regression
3. Random Forest
4. XGBoost

Evaluation:
- MAE
- RMSE
- R²


In [ ]:

def evaluate_predictions(name, y_true, predictions):
    predictions = np.maximum(predictions, 0)

    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": mean_squared_error(y_true, predictions) ** 0.5,
        "R2": r2_score(y_true, predictions),
    }


results = []
predictions_dict = {}

baseline_pred = np.repeat(
    y_train.mean(),
    len(y_test)
)

results.append(
    evaluate_predictions(
        "Mean Baseline",
        y_test,
        baseline_pred,
    )
)

predictions_dict["Mean Baseline"] = baseline_pred


In [ ]:

linear_model = LinearRegression()
linear_model.fit(X_train, y_train_log)

linear_pred_log = linear_model.predict(X_test)
linear_pred = np.expm1(linear_pred_log)
linear_pred = np.maximum(linear_pred, 0)

results.append(
    evaluate_predictions(
        "Linear Regression",
        y_test,
        linear_pred,
    )
)

predictions_dict["Linear Regression"] = linear_pred


In [ ]:

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train_log)

rf_pred_log = rf_model.predict(X_test)
rf_pred = np.expm1(rf_pred_log)
rf_pred = np.maximum(rf_pred, 0)

results.append(
    evaluate_predictions(
        "Random Forest",
        y_test,
        rf_pred,
    )
)

predictions_dict["Random Forest"] = rf_pred


In [ ]:

xgb_model = XGBRegressor(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train_log)

xgb_pred_log = xgb_model.predict(X_test)
xgb_pred = np.expm1(xgb_pred_log)
xgb_pred = np.maximum(xgb_pred, 0)

results.append(
    evaluate_predictions(
        "XGBoost",
        y_test,
        xgb_pred,
    )
)

predictions_dict["XGBoost"] = xgb_pred


In [ ]:
model_metrics = (
    pd.DataFrame(results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(model_metrics)

best_overall_name = model_metrics.iloc[0]["Model"]

best_ml_model_name = (
    model_metrics[
        model_metrics["Model"] != "Mean Baseline"
    ]
    .iloc[0]["Model"]
)

print("Best overall result by MAE:", best_overall_name)
print("Best trained ML model by MAE:", best_ml_model_name)

baseline_mae = float(
    model_metrics.loc[
        model_metrics["Model"] == "Mean Baseline",
        "MAE",
    ].iloc[0]
)

best_ml_mae = float(
    model_metrics.loc[
        model_metrics["Model"] == best_ml_model_name,
        "MAE",
    ].iloc[0]
)

if best_ml_mae < baseline_mae:
    improvement = (
        (baseline_mae - best_ml_mae)
        / baseline_mae
        * 100
    )
    print(
        f"{best_ml_model_name} improves MAE over the mean baseline "
        f"by {improvement:.1f}%."
    )
else:
    print(
        "The trained ML models did not beat the mean baseline on MAE. "
        "This should be reported honestly rather than hidden."
    )


### Important interpretation note

Future customer spend is difficult to predict because many customers make no purchase in the target period and spending has a heavy-tailed distribution.

The goal is not to claim perfect forecasting. The value of this section is demonstrating:
- leakage-safe feature engineering,
- supervised regression,
- sensible baselines,
- model comparison,
- and honest evaluation.


## 20. Actual vs predicted spend

In [ ]:

best_predictions = predictions_dict[best_ml_model_name]

comparison = pd.DataFrame({
    "ActualSpend": y_test.values,
    "PredictedSpend": best_predictions,
})

plot_sample = comparison.copy()

# Clip only for visual readability. Metrics above use the un-clipped values.
x_cap = plot_sample["ActualSpend"].quantile(0.98)
y_cap = plot_sample["PredictedSpend"].quantile(0.98)

plt.figure(figsize=(7, 6))
plt.scatter(
    plot_sample["ActualSpend"].clip(upper=x_cap),
    plot_sample["PredictedSpend"].clip(upper=y_cap),
    alpha=0.5,
    s=20,
)
plt.title(f"Actual vs Predicted 90-Day Spend — {best_ml_model_name}")
plt.xlabel("Actual Spend")
plt.ylabel("Predicted Spend")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 21. Feature importance

In [ ]:

if best_ml_model_name == "XGBoost":
    importance_values = xgb_model.feature_importances_
    importance_model = xgb_model

elif best_ml_model_name == "Random Forest":
    importance_values = rf_model.feature_importances_
    importance_model = rf_model

else:
    # For a non-tree winner, use permutation importance on the fitted linear model.
    importance_model = linear_model

    perm = permutation_importance(
        importance_model,
        X_test,
        np.log1p(y_test),
        n_repeats=10,
        random_state=RANDOM_STATE,
        scoring="neg_mean_absolute_error",
    )

    importance_values = perm.importances_mean


feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": importance_values,
}).sort_values("Importance", ascending=False)

display(feature_importance)

plot_imp = feature_importance.sort_values("Importance")

plt.figure(figsize=(9, 6))
plt.barh(plot_imp["Feature"], plot_imp["Importance"])
plt.title(f"Feature Importance — {best_ml_model_name}")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()



# Part D — Customer Prioritisation Layer

## 22. Combine segmentation and predicted future value

We now connect descriptive and predictive analytics:

- segment = how the customer has behaved historically;
- predicted spend = how much value the model expects over the next 90 days.

This makes the output more useful than clustering alone.


In [ ]:

# Refit the best-performing model on all modelling customers
# so every eligible customer receives a prediction.

y_all_log = np.log1p(y)

if best_ml_model_name == "XGBoost":
    final_model = XGBRegressor(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

elif best_ml_model_name == "Random Forest":
    final_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

else:
    final_model = LinearRegression()

final_model.fit(X, y_all_log)

all_pred = np.expm1(final_model.predict(X))
all_pred = np.maximum(all_pred, 0)

model_data["Predicted90DaySpend"] = all_pred

customer_analytics = rfm[
    [
        "Customer ID",
        "Recency",
        "Frequency",
        "Monetary",
        "Cluster",
        "Segment",
    ]
].merge(
    model_data[
        ["Customer ID", "Predicted90DaySpend"]
    ],
    on="Customer ID",
    how="left",
)

display(
    customer_analytics.sort_values(
        "Predicted90DaySpend",
        ascending=False,
    ).head(20)
)


In [ ]:

# Simple prioritisation examples

prediction_threshold = (
    customer_analytics["Predicted90DaySpend"].quantile(0.75)
)

customer_analytics["PriorityGroup"] = "Standard"

customer_analytics.loc[
    (
        customer_analytics["Segment"] == "Recent / Developing"
    )
    & (
        customer_analytics["Predicted90DaySpend"] >= prediction_threshold
    ),
    "PriorityGroup",
] = "Emerging High-Value"

customer_analytics.loc[
    customer_analytics["Segment"] == "Lapsed Valuable Customers",
    "PriorityGroup",
] = "Re-engagement Opportunity"

customer_analytics.loc[
    customer_analytics["Segment"] == "High-Value Loyal",
    "PriorityGroup",
] = "Protect / Retain"

priority_summary = (
    customer_analytics.groupby("PriorityGroup")
    .agg(
        Customers=("Customer ID", "count"),
        AvgHistoricalSpend=("Monetary", "mean"),
        AvgPredicted90DaySpend=("Predicted90DaySpend", "mean"),
    )
    .sort_values(
        "AvgPredicted90DaySpend",
        ascending=False,
    )
)

display(priority_summary)



## 23. Business interpretation framework

The algorithms do not prescribe marketing actions automatically.  
The following are reasonable interpretations of the resulting customer groups:

- **High-Value Loyal** → retention, loyalty benefits, premium cross-sell.
- **Recent / Developing** → encourage repeat purchases and develop the relationship.
- **Lapsed Valuable Customers** → targeted re-engagement.
- **Low-Engagement / Dormant** → low-cost reactivation rather than expensive retention.

The predictive model adds a second dimension by identifying customers expected to generate relatively high future value.


## 24. Export all project outputs

In [ ]:

OUTPUT_DIR = "retailiq_advanced_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

customer_analytics.to_csv(
    os.path.join(OUTPUT_DIR, "customer_analytics.csv"),
    index=False,
)

cluster_summary.to_csv(
    os.path.join(OUTPUT_DIR, "cluster_summary.csv"),
    index=False,
)

k_evaluation.to_csv(
    os.path.join(OUTPUT_DIR, "k_evaluation.csv"),
    index=False,
)

retention.to_csv(
    os.path.join(OUTPUT_DIR, "cohort_retention.csv")
)

model_metrics.to_csv(
    os.path.join(OUTPUT_DIR, "model_metrics.csv"),
    index=False,
)

feature_importance.to_csv(
    os.path.join(OUTPUT_DIR, "feature_importance.csv"),
    index=False,
)

model_data.to_csv(
    os.path.join(OUTPUT_DIR, "predictive_model_dataset.csv"),
    index=False,
)

zip_path = "RetailIQ_Advanced_outputs.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED,
) as zf:
    for filename in os.listdir(OUTPUT_DIR):
        zf.write(
            os.path.join(OUTPUT_DIR, filename),
            arcname=filename,
        )

print("Saved outputs:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", os.path.join(OUTPUT_DIR, filename))

print(" -", zip_path)


In [ ]:

# Optional automatic download when using Google Colab.

try:
    from google.colab import files
    files.download("RetailIQ_Advanced_outputs.zip")
except ImportError:
    print(
        "Not running in Colab. "
        "The ZIP is available in the current working directory."
    )



# Final Project Summary

## Problem
The retailer has more than one million transaction records but no predefined customer segments or future customer-value estimates.

## Approach

### Customer segmentation
- cleaned cancellations, invalid purchases and missing customer IDs;
- engineered RFM features;
- treated extreme values and skew;
- standardised the features;
- evaluated K using Elbow and Silhouette diagnostics;
- used K-Means to create four customer groups;
- used PCA for visualisation.

### Customer lifecycle
- created monthly acquisition cohorts;
- measured customer retention over subsequent months.

### Predictive ML
- created a leakage-safe 90-day future-spend target;
- engineered customer behavioural features;
- compared a mean baseline, Linear Regression, Random Forest and XGBoost;
- evaluated models using MAE, RMSE and R²;
- inspected feature importance.

### Decision layer
- combined historical customer segments with predicted future spend;
- created simple customer-prioritisation categories.

---

# Interview Questions You Should Be Able to Answer

### Why RFM?
It compresses detailed transaction history into three intuitive behavioural dimensions: recency, frequency and monetary value.

### Why standardise before K-Means?
K-Means relies on Euclidean distance. Without scaling, a numerically large feature such as Monetary can dominate the clustering.

### Why log-transform RFM?
Purchase frequency and spend are highly right-skewed. Log transformation reduces the effect of extreme values and makes distance-based clustering more stable.

### Why choose K=4 if another K has a higher silhouette score?
Silhouette score is one diagnostic, not the only objective. K=4 provides richer and more actionable behavioural groups. The trade-off should be stated explicitly.

### Why PCA?
Only to visualise the 3-dimensional scaled RFM feature space in 2 dimensions. PCA does not create the customer labels in this notebook.

### Why use a time cut-off for future-spend prediction?
It prevents leakage. Features are created only from data available before the prediction window.

### Why use a mean baseline?
A predictive model should outperform a simple reference strategy. Otherwise the additional model complexity may not be justified.

### Why train on log future spend?
Future spending is heavily right-skewed. The log transformation reduces the impact of extreme values while evaluation remains on the original monetary scale.

### Why compare several models?
Linear Regression provides an interpretable baseline, while Random Forest and XGBoost capture nonlinear relationships and feature interactions.

### What are the limitations?
- RFM ignores product semantics, demographics and marketing exposure.
- K-Means assumes distance-based cluster structure.
- The future-spend target contains many zero-spend customers.
- The predictive evaluation uses one historical/future cut-off rather than multiple rolling-origin backtests.
- Business actions are analytical recommendations, not experimentally proven causal effects.
